In [ ]:

import matplotlib
import platform

#For making sure the animation works on linux systems
if platform.system() == 'Linux':
    matplotlib.use('TkAgg')
    
#Importamos las librerias
import numpy as np
import random 
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
#This function is used to measure distances
def medir_distancia(ruta, matriz_dist):
    dist=0
    for i in range(len(ruta)-1):
        #We sum the distances between cities
        dist += matriz_dist[ruta[i], ruta[i+1]]
    #From the last city we go back to the first
    dist += matriz_dist[ruta[-1], ruta[0]]
    return dist

In [ ]:
#We define at random the cities
n_ciudades=20

pos_ciudades=np.random.rand(n_ciudades, 2) * 100
# We define a distance matrix
matriz_dist = np.linalg.norm(pos_ciudades[:, np.newaxis] - pos_ciudades, axis=2)

In [ ]:
#Function to solve and animate the TSP problem
def resolver_tsp_an(ciudades, temp_inicial, enfriamiento, pasos):
    
    n = len(ciudades)

    #We generate a random route
    ruta_actual = list(range(n))
    random.shuffle(ruta_actual)
    #Distance of the initial route
    mejor_distancia = medir_distancia(ruta_actual, matriz_dist)

    mejor_ruta = ruta_actual[:]
    temp = temp_inicial

    #We save the routes and distances in order to animate
    rutas_an = []
    distancias_an = []

    #Montecarlo algorithm to solve the problem
    for i in range(pasos):
        #We swap 2 cities at random
        ruta_nueva = ruta_actual[:]

        i_1, i_2 = random.sample(range(n), 2)
     
        ruta_nueva[i_1], ruta_nueva[i_2] = ruta_nueva[i_2], ruta_nueva[i_1]
        #Calculate the new distance
        distancia_nueva = medir_distancia(ruta_nueva, matriz_dist)

        #Metropolis-Hastings criteria
        if distancia_nueva < mejor_distancia or random.random() < np.exp((mejor_distancia - distancia_nueva) / temp):
            ruta_actual = ruta_nueva
            mejor_distancia = distancia_nueva
            #
            if distancia_nueva < medir_distancia(mejor_ruta, matriz_dist):
                mejor_ruta = ruta_nueva[:]
            #Cooling the system to ensure the convergence
            temp *= enfriamiento

        #We save the route
        if i % 50 == 0:
            rutas_an.append(ruta_actual[:])
            distancias_an.append(medir_distancia(ruta_actual, matriz_dist))

    return mejor_ruta, mejor_distancia, rutas_an, distancias_an


In [5]:
ruta_optima, dist_final, rutas, distancias=resolver_tsp_an(pos_ciudades, 10000, 0.990, 50000)

In [ ]:
#Animation function
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6)) 

#Gráfica de la ruta
ax1.set_title("Evolución de la Ruta del TSP")
ax1.scatter(pos_ciudades[:, 0], pos_ciudades[:, 1], c='red', marker='o', label='Ciudades')
line, = ax1.plot([], [], 'b-o', lw=2)
start_node, = ax1.plot([], [], 'go', markersize=10, label='Inicio')

#Gráfica de la distancia
ax2.set_title("Distancia Total de la Ruta")
ax2.set_xlabel("Paso de Animación")
ax2.set_ylabel("Distancia")
dist_line, = ax2.plot([], [], 'r-')
dist_text = ax2.text(0.05, 0.95, '', transform=ax2.transAxes, fontsize=12, verticalalignment='top') 

def init():
    line.set_data([], [])
    start_node.set_data([], [])
    dist_line.set_data([], [])
    dist_text.set_text('')
    ax1.legend()
    return line, start_node, dist_line, dist_text

def update(frame):
    #Actualizar la gráfica de la ruta
    ruta = rutas[frame]
    x_coords = pos_ciudades[ruta + [ruta[0]], 0] #Añadir el punto de inicio al final para cerrar el circuito
    y_coords = pos_ciudades[ruta + [ruta[0]], 1]
    line.set_data(x_coords, y_coords)
    
    #Marcar el nodo de inicio
    start_node.set_data(pos_ciudades[ruta[0], 0], pos_ciudades[ruta[0], 1])

    #Actualizar la gráfica de la distancia
    dist_line.set_data(range(frame + 1), distancias[:frame + 1])
    dist_text.set_text(f'Distancia: {distancias[frame]:.2f}')
    
    ax2.set_xlim(0, len(distancias))
    ax2.set_ylim(min(distancias) * 0.9, max(distancias) * 1.1)

    return line, start_node, dist_line, dist_text

#Crear la animación
ani = FuncAnimation(fig, update, frames=len(rutas),
                    init_func=init, blit=False, interval=1) #interval in ms (50ms = 20 fps)

plt.tight_layout()
plt.show() # Mostrar la animación